In [0]:
%pip install pymysql

In [0]:
from pyspark.sql.functions import *
from framework.reader import *
from framework.writer import *
from framework.audit import *
from framework.validator import *
from framework.metrics import *
from framework.mysql_connector import *
from framework.transformations.gold_transformations.gold_transformation import *

In [0]:
Gold_Transformations={"build_daily_product_sales":build_daily_product_sales,
                      "build_customer_sales_summary":build_customer_sales_summary,
                      "build_product_performance":build_product_performance,
                      "build_daily_sales_summary":build_daily_sales_summary}

In [0]:
gold_tables=fetch_eligible_gold_pipelines()
for table in gold_tables:
    dataframes={}
    sources=fetch_eligible_gold_pipelines_source(table["gold_pipeline_id"])
    for source in sources:
        dataframes[source["source_name"]]=read_delta(spark,source["source_path"])
    transform=Gold_Transformations[table["transformation_fun"]]
    gold_df=transform(dataframes)
    write_delta(gold_df,table["target_path"],table["write_mode"])